In [ ]:
from ultralytics import YOLO

# Load a model
model = YOLO("yolo11n.pt")

# Train the model using our custom dataset.yaml
train_results = model.train(
    data="dataset.yaml",  # path to our custom dataset YAML
    epochs=50,           # number of training epochs
    imgsz=640,            # training image size
    device="cpu",         # specify device or use GPU if available
    patience=10          # early stopping patience
)

# Evaluate model performance on the validation set
metrics = model.val()

# Perform object detection on an image
results = model("dataset_yolo/Screenshot 2025-02-10 104720_aug_0.png")
results[0].show()

# Export the model to ONNX format
path = model.export(format="onnx")  # return path to exported model

New https://pypi.org/project/ultralytics/8.3.82 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.74  Python-3.11.9 torch-2.6.0+cpu CPU (AMD Ryzen 5 5600X 6-Core Processor)
engine\trainer: task=detect, mode=train, model=../runs/detect/train5/weights/best.pt, data=dataset.yaml, epochs=50, time=None, patience=10, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=cpu, workers=8, project=None, name=train6, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show

train: Scanning C:\Users\Knut\OneDrive\Documents\GitHub\rotmg-projectile-tracking\knut_sandbox\dataset_yolo_whole\training.cache... 220 images, 0 backgrounds, 0 corrupt: 100%|██████████| 220/220 [00:00<?, ?it/s]
val: Scanning C:\Users\Knut\OneDrive\Documents\GitHub\rotmg-projectile-tracking\knut_sandbox\dataset_yolo_whole\validation.cache... 51 images, 0 backgrounds, 0 corrupt: 100%|██████████| 51/51 [00:00<?, ?it/s]


Plotting labels to c:\Users\Knut\OneDrive\Documents\GitHub\rotmg-projectile-tracking\runs\detect\train6\labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 81 weight(decay=0.0), 88 weight(decay=0.0005), 87 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to c:\Users\Knut\OneDrive\Documents\GitHub\rotmg-projectile-tracking\runs\detect\train6
Starting training for 50 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


       1/50         0G     0.8525     0.5423     0.8844         47        640:  21%|██▏       | 3/14 [00:15<00:55,  5.04s/it]


KeyboardInterrupt: 

In [5]:
path = model.export(format="onnx")

Ultralytics 8.3.74  Python-3.11.9 torch-2.6.0+cpu CPU (AMD Ryzen 5 5600X 6-Core Processor)

PyTorch: starting from 'c:\Users\Knut\OneDrive\Documents\GitHub\rotmg-projectile-tracking\runs\detect\train3\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (5.2 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0', 'onnxslim', 'onnxruntime'] not found, attempting AutoUpdate...
   ---------------------------------------- 14.5/14.5 MB 65.1 MB/s eta 0:00:00
   ---------------------------------------- 11.3/11.3 MB 79.1 MB/s eta 0:00:00

requirements: AutoUpdate success  27.7s, installed 3 packages: ['onnx>=1.12.0', 'onnxslim', 'onnxruntime']
requirements:  Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.17.0 opset 19...
ONNX: slimming with onnxslim 0.1.48...
ONNX: export success  30.0s, saved as 'c:\Users\Knut\OneDrive\Documents\GitHub\rotmg-projectile-tracking\runs\detect\train3\weights\best.onnx' (10.1 

In [5]:
import cv2
import time
import numpy as np
from PIL import Image
from ultralytics import YOLO

def preprocess_frame(frame):
    # Convert frame from BGR (OpenCV default) to RGB
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    pil_image = Image.fromarray(frame_rgb)
    
    # First crop: according to processing code (assumed frame is 2560x1440)
    crop_box_1 = (0, 0, 2080, 1440)
    image_cropped = pil_image.crop(crop_box_1)
    
    # Second crop: crop equally from left and right to obtain a 1440x1440 square
    crop_box_2 = (320, 0, 320 + 1440, 1440)
    final_image = image_cropped.crop(crop_box_2)
    
    # Resize to 640x640 (model input size)
    img_resized = final_image.resize((640, 640))
    
    # Convert back to BGR numpy array for OpenCV
    processed_frame = cv2.cvtColor(np.array(img_resized), cv2.COLOR_RGB2BGR)
    return processed_frame

# Load your trained model with weights
model = YOLO("../runs/detect/train5/weights/best.pt")

# Specify the path to your video file (or 0 for webcam)
video_path = r"D:\Vids\Desktop\Desktop 2025.02.10 - 10.43.42.23.DVR.mp4"  # Update as needed
cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    print("Error opening video stream or file")
    exit()

frame_total = 0
frame_count = 0
total_inference_time = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_total += 1
    # Process only every second frame for performance.
    if frame_total % 2 != 0:
        continue

    processed_frame = preprocess_frame(frame)
    
    start_time = time.time()
    # Set confidence threshold to 0.6 to filter out weaker detections.
    results = model(processed_frame, conf=0.6)
    end_time = time.time()

    inference_time = end_time - start_time
    total_inference_time += inference_time
    frame_count += 1

    # Annotate the frame with detections (draw bounding boxes)
    annotated_frame = results[0].plot()  # returns an image with drawn boxes

    # Calculate instantaneous FPS and annotate
    fps = 1 / inference_time if inference_time > 0 else 0
    cv2.putText(annotated_frame, f"FPS: {fps:.2f}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    cv2.imshow("Detection", annotated_frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

if frame_count > 0:
    avg_fps = frame_count / total_inference_time
    print(f"Processed {frame_count} frames. Average FPS: {avg_fps:.2f}")


0: 640x640 (no detections), 71.5ms
Speed: 2.6ms preprocess, 71.5ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 72.3ms
Speed: 1.5ms preprocess, 72.3ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 70.1ms
Speed: 0.5ms preprocess, 70.1ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 68.3ms
Speed: 1.0ms preprocess, 68.3ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 57.7ms
Speed: 1.0ms preprocess, 57.7ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 60.2ms
Speed: 2.5ms preprocess, 60.2ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 60.2ms
Speed: 1.5ms preprocess, 60.2ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 78.6ms
Speed: 2.0ms preprocess, 78.6ms i